# Held-out evaluation -- `v2_ctc_ar_sequential`

Standalone evaluator for the checkpoint produced by
`train_recognizer_ctc_ar_sequential.ipynb`. Rebuilds the **exact** held-out
validation split that run used (same shuffle seed, same `val_frac`, same
CTC-unlearnable filter, same newline-filtered dataset), then reports
**per-source** Character Error Rate and exact-match accuracy for both heads:

- **AR** -- the decoder's autoregressive greedy read-out (`decode_greedy_sequential`).
- **CTC** -- the encoder's alignment-free greedy read-out.

Sections: **3b** held-out val, **4** external unseen set, **5** the separate
CTC-only ablation checkpoint (for comparison against 3b's CTC column).

Runs on Colab / Kaggle / local, CPU / GPU / TPU (auto-detected). Sections 1a
and 1b are the same tokenizer-decode and low-memory patches the training
notebook applies -- 1a matters here because without it real spaces are
stripped from AR output before CER is computed, inflating the number.

**Prereqs:** an `HF_TOKEN` secret with read access to `Panhapich/Tuna-OCR`
(checkpoints) and `Panhapich/tuna-ocr-data` (the prebuilt dataset). If the
dataset has never been built, run the DATA cells of the training notebook once first.

## 1. Setup

In [9]:
import os, subprocess, sys

def detect_environment():
    # Kaggle is checked first: some Kaggle kernels leak a stray COLAB_GPU/
    # COLAB_RELEASE_TAG env var, which would otherwise misdetect as Colab and
    # crash trying to mount Google Drive. "/kaggle/working" existing is a much
    # harder signal to spoof than an env var, so it takes priority.
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.isdir("/kaggle/working"):
        return "kaggle"
    if "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ:
        return "colab"
    return "local"

ENV = detect_environment()
print("environment:", ENV)

REPO_URL = "https://github.com/Pich09/tuna-ocr.git"
REPO_DIR = "tuna-ocr"

def run_git(args):
    """Runs git and raises with git's ACTUAL stderr on failure. A bare
    CalledProcessError only reports "exit status 128", which is git's catch-all
    and says nothing about which of the many possible causes (existing
    directory, auth, network) actually happened."""
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed ({r.returncode}):\n{r.stderr.strip()}")
    return r

# Three cases, in order. The middle one is the important fix: after a kernel
# restart the cwd resets to /content (or /kaggle/working), so "recognizer" is no
# longer visible even though a previous run already cloned the repo -- the old
# code then tried to clone again and git aborted with "destination path already
# exists and is not an empty directory" (exit 128).
def pull_latest():
    """Force the clone we're standing in to exactly match origin/main, LOUDLY.
    A stale/diverged clone is the single most confusing failure mode of this
    notebook: the library code is older (or locally modified) vs. what the
    notebook cell driving it assumes, so you get e.g. a TypeError about an
    unexpected keyword argument for something that plainly exists on GitHub.

    This used to be `git pull --ff-only`, which FAILS SILENTLY (just a printed
    warning, not raised) whenever the local clone has diverged from a clean
    fast-forward -- and it always does on Colab/Kaggle, because "restart
    runtime" only restarts the Python kernel, not the filesystem: /content
    (or /kaggle/working) persists across restarts, so a previous session's
    leftover local state (a stray edit, a half-finished git operation, a
    detached HEAD from checking out a specific commit while debugging) sticks
    around and silently blocks every future pull -- so restarting the runtime
    looks like it did nothing, over and over, even though the kernel really
    did restart. This is an ephemeral, notebook-driven clone with no local
    changes ever worth preserving, so there is no real downside to a hard
    reset -- fetch + reset --hard is unconditional and can't get stuck the
    way a fast-forward-only pull can."""
    try:
        run_git(["fetch", "origin"])
        run_git(["reset", "--hard", "origin/main"])
        run_git(["clean", "-fd"])
        print("reset to latest origin/main")
    except RuntimeError as e:
        print("!" * 78)
        print("WARNING: could not update the clone -- running POSSIBLY STALE code.")
        print(f"  {e}")
        print("  If a later cell fails with 'unexpected keyword argument', this is why.")
        print("  This is likely a network issue (offline runtime) since the reset above")
        print("  doesn't fail on local divergence anymore -- check connectivity.")
        print("!" * 78)

if os.path.isdir("recognizer"):
    # Already inside the repo -- which is what re-running this cell in the same
    # session always looks like, since the first run chdir'd here. This branch
    # used to just print and return, so a second run silently kept whatever code
    # the session started with and never saw upstream commits again.
    print(f"already inside the repo working dir: {os.getcwd()}")
    pull_latest()
elif os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    print(f"found an existing clone, reusing it: {os.getcwd()}")
    pull_latest()
else:
    run_git(["clone", REPO_URL, REPO_DIR])
    os.chdir(REPO_DIR)
    print(f"cloned into {os.getcwd()}")

sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())
# Print the resolved commit: the one unambiguous answer to "is my library code
# actually the version I think it is?", checkable against the GitHub history.
print("repo commit:  " + run_git(["log", "-1", "--pretty=%h %s"]).stdout.strip())


environment: colab
already inside the repo working dir: /content/tuna-ocr
pulled latest changes
working dir: /content/tuna-ocr
repo commit:  3ee70b3 AR: add windowed repetition penalty to decode_greedy(_sequential)


In [10]:
# Colab and Kaggle both ship torch preinstalled and matched to their runtime (the CUDA
# driver on a GPU runtime, or the torch_xla/libtpu build on a TPU runtime) -- blindly
# `pip install torch` on top of that (e.g. via a plain `-r recognizer/requirements.txt`)
# can silently replace it with a build that doesn't match, which breaks GPU support and
# breaks TPU support even harder (torch_xla is pinned to one exact torch version).
# Install everything else normally, and only pip-install torch if it isn't importable
# at all (a bare local venv).
import importlib.util
from pathlib import Path

def strip_torch(req_path):
    lines = Path(req_path).read_text().splitlines()
    return [l for l in lines if not l.strip().lower().startswith("torch")]

torch_before = None
if importlib.util.find_spec("torch") is not None:
    import torch
    torch_before = torch.__version__

reqs = strip_torch("recognizer/requirements.txt") + strip_torch("real_data/requirements.txt")
Path("/tmp/_notebook_requirements.txt").write_text("\n".join(reqs) + "\n")
!pip install -q -r /tmp/_notebook_requirements.txt

if torch_before is None:
    print("torch not found -- installing (no preinstalled build to preserve here)")
    !pip install -q torch
else:
    # Excluding torch from the requirements file isn't a complete guarantee: any
    # dependency in it is free to pull a *different* torch in as its own dependency.
    # On a TPU runtime that's silently fatal -- torch_xla only loads against the exact
    # torch build it was compiled for, and the failure surfaces much later as an opaque
    # import/libtpu error, so check explicitly here rather than discovering it then.
    # importlib.metadata, not `torch.__version__`: torch is already imported in this
    # kernel, so its module object still reports the OLD version no matter what pip
    # just wrote to disk (and importlib.reload(torch) is not a safe way to find out).
    # The distribution metadata reflects what's actually installed now.
    from importlib.metadata import version as _pkg_version
    torch_after = _pkg_version("torch")
    if torch_after != torch_before:
        print(f"WARNING: pip changed torch {torch_before} -> {torch_after} as a "
              f"transitive dependency. On a TPU runtime, restart the runtime and "
              f"`pip install torch=={torch_before}` before continuing, or torch_xla "
              f"will fail to load.")
    else:
        print(f"using preinstalled torch {torch_after} "
              f"(cuda available: {torch.cuda.is_available()}) -- not reinstalled")


using preinstalled torch 2.11.0+cpu (cuda available: False) -- not reinstalled


In [11]:
import os

from recognizer import env_utils

checkpoint_root = env_utils.get_checkpoint_root(ENV)

# Token resolution, in order. Colab's own secret store (env_utils.get_hf_token) is
# tried first but is NOT reliable: it raises "Secrets can only be fetched when running
# from the Colab UI" whenever the notebook runs detached from the UI tab, which is
# exactly what happened on a long training run here. So fall back to an HF_TOKEN
# environment variable, then to a plain file, then to an interactive prompt --
# deliberately never hardcoded in this notebook, which is committed to a public git
# repo (GitHub's push protection rejects the commit outright, and HF's secret scanner
# auto-revokes any write-scoped token that lands in one).
#
# Easiest on Colab: run this in a scratch cell once per session, paste when prompted:
#     import os, getpass; os.environ["HF_TOKEN"] = getpass.getpass("HF token: ")
hf_token = None
try:
    hf_token = env_utils.get_hf_token(ENV)
except Exception as e:
    print(f"platform secret store unavailable ({type(e).__name__}), trying fallbacks...")
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    for candidate in ("/content/hf_token.txt", "/kaggle/working/hf_token.txt", "hf_token.txt"):
        if os.path.exists(candidate):
            hf_token = open(candidate).read().strip()
            print(f"read HF token from {candidate}")
            break
if not hf_token:
    import getpass
    hf_token = getpass.getpass("HF token (input hidden): ").strip()

# Resolve the accelerator NOW, before the multi-hour cells below, and print the exact
# torch device that training will use. detect_accelerator() covers both TPU
# generations (legacy XRT env vars and current PJRT ones) plus the /dev/accel* device
# nodes, so a modern Colab/Kaggle TPU runtime is recognised rather than falling through
# to CPU -- a fallback that is otherwise invisible until you notice steps taking 100x
# too long, hours in.
accelerator = env_utils.detect_accelerator()
device = env_utils.get_torch_device()

print("environment:      ", ENV)
print("checkpoint root:  ", checkpoint_root)
print("HF token loaded:  ", bool(hf_token))
print("accelerator:      ", env_utils.describe_accelerator())
print("torch device:     ", device)
if accelerator == "cpu":
    print("\n>>> No GPU/TPU detected. Colab: Runtime > Change runtime type. "
          "Kaggle: Settings > Accelerator. Do not start the training cell on CPU -- "
          "at this dataset's scale it will not finish.")


platform secret store unavailable (RuntimeError), trying fallbacks...
environment:       colab
checkpoint root:   /content/drive/My Drive/tuna-ocr/checkpoints
HF token loaded:   True
accelerator:       cpu (no GPU/TPU detected -- training will be impractically slow at this scale)
torch device:      cpu

>>> No GPU/TPU detected. Colab: Runtime > Change runtime type. Kaggle: Settings > Accelerator. Do not start the training cell on CPU -- at this dataset's scale it will not finish.


In [12]:
# Downloads Panhapich/khmer-sp-8k's SentencePiece model + khmer_segmentation.py
# wrapper (a bare .model file is not enough -- see recognizer/README.md).
from recognizer.tokenizer.fetch_tokenizer import fetch_tokenizer

fetch_tokenizer()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetched Panhapich/khmer-sp-8k -> /content/tuna-ocr/recognizer/tokenizer/assets: ['gazetteer.json', 'khmer_segmentation.py', 'khmer_sp.model', 'latin_exceptions.json', 'tokenizer_info.json']


PosixPath('/content/tuna-ocr/recognizer/tokenizer/assets')

## 1a. Patch a tokenizer decode bug (inflates val_ar_cer)

The vendored `khmer_segmentation.py` (downloaded above from `Panhapich/khmer-sp-8k`)
has a `KhmerTokenizer.decode()` that does `self.sp.decode(ids).replace(" ", "")` --
this strips **every** space unconditionally, not just the artificial Khmer
word-boundary spaces `segment_line()` introduces for SentencePiece training. So any
real space in the reference text (English words, mixed Khmer/English text,
punctuation spacing) is deleted from the AR decoder's decoded output regardless of
whether the model predicted it correctly -- inflating `val_ar_cer` for any sample
containing genuine whitespace. `val_ctc_cer` is unaffected (`CharVocab.decode` in
`recognizer/data/char_vocab.py` does no space stripping), so it remains a trustworthy
read on encoder accuracy even without this patch.

This cell rewrites the downloaded `khmer_segmentation.py` on disk (the same file
every `KhmerOcrTokenizer()` construction loads from -- see
`khmer_ocr_tokenizer.py`'s `_load_upstream_khmer_tokenizer`) so `decode()` only
strips a space when it falls strictly between two Khmer characters -- the actual
artificial-boundary case -- and leaves every other space alone. Idempotent: skips if
already patched, so re-running this cell (or a fresh fetch that re-downloads the
original file) is safe.

In [13]:
from recognizer.config import TOKENIZER_ASSETS_DIR

_seg_path = TOKENIZER_ASSETS_DIR / "khmer_segmentation.py"
_seg_src = _seg_path.read_text(encoding="utf-8")

_BUGGY_DECODE = '''    def decode(self, ids) -> str:
        # Strip the artificial word-boundary spaces introduced for training;
        # natural Khmer orthography does not space every word.
        return self.sp.decode(ids).replace(" ", "")'''

_PATCHED_DECODE = '''    def decode(self, ids) -> str:
        # PATCHED (notebook cell 1a): the original body here did
        # `self.sp.decode(ids).replace(" ", "")`, which strips EVERY space --
        # including genuine ones in English words, mixed Khmer/English text, and
        # punctuation spacing -- not just the artificial Khmer word-boundary
        # spaces segment_line() introduces for SentencePiece training. That
        # silently deletes correctly-predicted spaces before CER ever sees them.
        # Only strip a space strictly between two Khmer characters -- the real
        # artificial-boundary case -- and leave every other space alone.
        import re as _re
        decoded = self.sp.decode(ids)
        return _re.sub(r"(?<=[\\u1780-\\u17ff])\\s(?=[\\u1780-\\u17ff])", "", decoded)'''

if _PATCHED_DECODE in _seg_src:
    print(f"{_seg_path} already patched -- nothing to do")
elif _BUGGY_DECODE in _seg_src:
    _seg_path.write_text(_seg_src.replace(_BUGGY_DECODE, _PATCHED_DECODE), encoding="utf-8")
    print(f"patched {_seg_path}: decode() now only strips Khmer-Khmer boundary spaces")
else:
    raise RuntimeError(
        f"{_seg_path} doesn't match the expected buggy decode() body -- the upstream "
        f"file may have changed. Inspect it manually before training: the goal is a "
        f"decode() that doesn't strip every space unconditionally."
    )


/content/tuna-ocr/recognizer/tokenizer/assets/khmer_segmentation.py already patched -- nothing to do


## 1b. Low-memory dataset loading patch

`recognizer/data/manifest.py`'s `load_dedup_arrow` (unmodified library code)
materializes the ENTIRE image-bytes column into a Python list
(`table.column("image").to_pylist()`), then builds a second full list of
`Sample` objects from it -- both lists stay alive simultaneously until the
function returns, so peak memory during dataset loading is roughly **2x**
the dataset's actual image-bytes size. On a Colab session this is enough to
get the kernel OOM-killed mid-load, which surfaces as no Python traceback at
all -- just `"Canceled future for execute_request message before replies
were done"` -- because the process itself dies, not one call inside it.

This cell monkeypatches `load_dedup_arrow` to iterate the Arrow columns
directly instead of pre-snapshotting them, so no intermediate full-column
Python list is ever alive alongside the final result -- same output, roughly
half the peak memory. This is a real fix to shared library code, not a
notebook-only workaround -- worth upstreaming into
`recognizer/data/manifest.py` directly once confirmed, so every consumer of
`run_training` benefits, not just this notebook.

In [14]:
# Monkeypatches recognizer.data.manifest.load_dedup_arrow: safe because
# load_dedup_manifest (which run_training actually calls) looks up
# load_dedup_arrow by name in the module's own namespace at CALL time, not at
# import time -- so reassigning the module attribute here takes effect for
# every call made after this cell runs, without needing to touch train.py or
# re-import anything downstream.
import recognizer.data.manifest as _manifest

def _load_dedup_arrow_low_memory(path):
    import pyarrow as pa

    with pa.memory_map(str(path), "rb") as source:
        table = pa.ipc.open_file(source).read_all()
    text_col = table.column("text")
    source_col = table.column("source")
    image_col = table.column("image")
    # zip() over ChunkedArrays iterates chunk-by-chunk, yielding pa.Scalar
    # objects one at a time -- .as_py() converts just that one value, so at
    # most one row's worth of extra Python objects exists beyond the `samples`
    # list actually being built, vs. the original's three full-column lists
    # PLUS the final list all alive at once.
    samples = []
    for t, s, img in zip(text_col, source_col, image_col):
        samples.append(_manifest.Sample(image_bytes=img.as_py(), text=t.as_py(), source=s.as_py()))
    return samples

_manifest.load_dedup_arrow = _load_dedup_arrow_low_memory
print("patched recognizer.data.manifest.load_dedup_arrow for lower peak memory during dataset loading")

patched recognizer.data.manifest.load_dedup_arrow for lower peak memory during dataset loading


## 2. Data

Downloads the prebuilt `dedup.arrow` from the Hub (built once by the training
notebook) and re-applies the **same** embedded-newline filter that notebook's
section 2c uses, producing `dedup_filtered.arrow` -- the exact file training
consumed. Both steps are cached: skipped if the files are already present and valid.

In [15]:
from pathlib import Path

import pyarrow as pa

from real_data import hf_push
from real_data.config import HF_DATA_REPO_ID, REAL_DATA_ROOT

dedup_raw = REAL_DATA_ROOT / "samples" / "dedup.arrow"
dedup_filtered = REAL_DATA_ROOT / "samples" / "dedup_filtered.arrow"


def _valid_arrow(p: Path) -> bool:
    if not p.exists():
        return False
    try:
        with pa.memory_map(str(p), "rb") as f:
            pa.ipc.open_file(f).schema
        return True
    except pa.ArrowInvalid:
        return False


# --- 2a. get dedup.arrow (download from the Hub if not already local) ---
if _valid_arrow(dedup_raw):
    print(f"{dedup_raw} present and valid -- skipping download")
elif hf_push.dataset_exists_on_hub(HF_DATA_REPO_ID, token=hf_token):
    print(f"downloading prebuilt dataset from {HF_DATA_REPO_ID} ...")
    hf_push.pull_dataset(dedup_raw, token=hf_token, repo_id=HF_DATA_REPO_ID)
    print("downloaded ->", dedup_raw)
else:
    raise RuntimeError(
        f"No local {dedup_raw} and nothing on {HF_DATA_REPO_ID}. Run the DATA "
        f"cells (section 2) of train_recognizer_ctc_ar_sequential.ipynb once to "
        f"build + push the dataset, then re-run this cell."
    )

# --- 2b. re-apply the training notebook's embedded-newline filter (its cell 2c) ---
if _valid_arrow(dedup_filtered):
    print(f"{dedup_filtered} present and valid -- skipping filter pass")
else:
    with pa.memory_map(str(dedup_raw), "rb") as source:
        table = pa.ipc.open_file(source).read_all()
    texts = table.column("text").to_pylist()
    sources = table.column("source").to_pylist()
    keep_mask = [("\n" not in t) for t in texts]
    dropped = {}
    for t, s, keep in zip(texts, sources, keep_mask):
        if not keep:
            dropped[s] = dropped.get(s, 0) + 1
    print(f"newline filter: dropping {len(keep_mask) - sum(keep_mask)}/{len(keep_mask)} "
          f"samples with embedded newlines: {dropped or 'none'}")
    filtered = table.filter(pa.array(keep_mask))
    tmp = dedup_filtered.with_name(dedup_filtered.name + ".tmp")
    with pa.OSFile(str(tmp), "wb") as sink:
        with pa.ipc.new_file(sink, filtered.schema) as writer:
            writer.write_table(filtered)
    tmp.replace(dedup_filtered)
    print(f"wrote {dedup_filtered}")

dedup_manifest = dedup_filtered
print("evaluation will use:", dedup_manifest)


/content/tuna-ocr/real_data/samples/dedup.arrow present and valid -- skipping download
/content/tuna-ocr/real_data/samples/dedup_filtered.arrow present and valid -- skipping filter pass
evaluation will use: /content/tuna-ocr/real_data/samples/dedup_filtered.arrow


## 3. Load checkpoint + shared machinery

Pulls `ctc_ar_sequential/best.pt` from `Panhapich/Tuna-OCR`, loads the model,
and defines `evaluate_groups(...)` -- the one driver used by both the held-out
eval (section 3b) and the external test set (section 4). Fast; run it once.
Results from 3b and 4 both append to `<checkpoint_root>/<RUN_NAME>/eval_results.csv`.

In [16]:
import csv
import io
import itertools
import random
import time
from collections import defaultdict
from pathlib import Path

import editdistance
import torch
from PIL import Image

from recognizer.config import TOKENIZER_ASSETS_DIR, TrainConfig
from recognizer.data.dataset import find_unlearnable
from recognizer.data.manifest import Sample, load_dedup_manifest
from recognizer.data.transforms import chunk_line_image, open_image
from recognizer.evaluate import load_model
from recognizer.hf_push import pull_best_checkpoint, pull_latest_checkpoint
from recognizer.modules.decoder import truncate_at_eos
from recognizer.tokenizer.khmer_ocr_tokenizer import KhmerOcrTokenizer

# ------------------------------- knobs ----------------------------------
CHECKPOINT_REPO_ID = "Panhapich/Tuna-OCR"
HUB_PATH_PREFIX = "ctc_ar_sequential"
RUN_NAME = "v2_ctc_ar_sequential"
WHICH = "best"                    # "best" or "latest"
BATCH_SIZE = 16
MAX_DECODE_LEN = 256              # matches training's decode_greedy_sequential default
REPETITION_PENALTY = 1.3          # CTRL-style; 1.0 = off (recognizer.modules.decoder's default)
REPETITION_WINDOW = 8             # only penalizes tokens from the last N generated positions
SHOW_WORST = 5                    # worst-by-AR-edit-distance examples printed per group
CSV_PATH = Path(checkpoint_root) / RUN_NAME / "eval_results.csv"
# --------------------------------------------------------------------

_tc = TrainConfig()
SEED, VAL_FRAC = _tc.seed, _tc.val_frac   # the values run_training actually used

ckpt_dir = Path(checkpoint_root) / RUN_NAME
ckpt_dir.mkdir(parents=True, exist_ok=True)
_pull = pull_best_checkpoint if WHICH == "best" else pull_latest_checkpoint
ckpt_path = _pull(ckpt_dir, token=hf_token, repo_id=CHECKPOINT_REPO_ID, path_prefix=HUB_PATH_PREFIX)
assert ckpt_path is not None, f"no {WHICH}.pt under {CHECKPOINT_REPO_ID}/{HUB_PATH_PREFIX}"
print("checkpoint:", ckpt_path)

tokenizer = KhmerOcrTokenizer(TOKENIZER_ASSETS_DIR)
model, model_cfg, char_vocab = load_model(ckpt_path, tokenizer, device)   # sets model.eval()
assert char_vocab is not None, "checkpoint has no char_vocab -- cannot decode CTC"


def scaled_width(s):
    with open_image(s.image_source) as im:
        w, h = im.size
    return max(1, round(w * model_cfg.img_height / max(1, h)))


def collapse_ctc(row, blank_id):
    out, prev = [], None
    for c in row:
        if c != prev and c != blank_id:
            out.append(c)
        prev = c
    return out


@torch.no_grad()
def decode_group(group, want_ar):
    """-> list[(ref, ar_hyp|None, ctc_hyp)]. Batched, width-sorted to limit padding
    (CER over a set is order-independent, so the sort is free)."""
    ordered = sorted(group, key=scaled_width)
    rows = []
    for i in range(0, len(ordered), BATCH_SIZE):
        batch = ordered[i:i + BATCH_SIZE]
        chunks, cpl = [], []
        for s in batch:
            ct, _ = chunk_line_image(s.image_source, model_cfg.chunk_width,
                                     model_cfg.chunk_overlap, model_cfg.img_height)
            chunks.extend(ct)
            cpl.append(len(ct))
        enc_out, enc_lengths, _ = model.encode(
            torch.stack(chunks).to(device), torch.tensor(cpl, dtype=torch.long))
        ctc_ids = model.ctc_head(enc_out).argmax(-1).cpu().tolist()
        lens = enc_lengths.cpu().tolist()
        ar_hyps = [None] * len(batch)
        if want_ar:
            ar_rows = model.decoder.decode_greedy_sequential(
                enc_out, enc_lengths, max_len=MAX_DECODE_LEN,
                repetition_penalty=REPETITION_PENALTY,
                repetition_window=REPETITION_WINDOW).tolist()
            ar_hyps = [tokenizer.decode(
                           truncate_at_eos(r, tokenizer.eos_id, tokenizer.pad_id),
                           strip_control=True)
                       for r in ar_rows]
        for s, crow, n, ah in zip(batch, ctc_ids, lens, ar_hyps):
            rows.append((s.text, ah,
                         char_vocab.decode(collapse_ctc(crow[:n], char_vocab.blank_id))))
    return rows


def cer(pairs):
    return sum(editdistance.eval(r, h) for r, h in pairs) / (sum(len(r) for r, _ in pairs) or 1)


def exact(pairs):
    return sum(r == h for r, h in pairs) / (len(pairs) or 1)


_HDR = (f"{'group':40}{'n_ar':>7}{'AR CER':>9}{'AR ex':>8}   "
        f"{'n_ctc':>7}{'CTC CER':>9}{'CTC ex':>8}")


def evaluate_groups(groups, ar_cap=None, ctc_cap=None, title=""):
    """groups: dict[label -> list[Sample]]. Prints a per-label table + micro-avg
    and returns csv rows [label, n_ar, ar_cer, ar_exact, n_ctc, ctc_cer, ctc_exact]."""
    all_ar, all_ctc, rows = [], [], []
    print(f"\n=== {title} ===" if title else "")
    print(_HDR)
    print("-" * len(_HDR))
    for label in sorted(groups):
        g = groups[label]
        ar_g = g if ar_cap is None else g[:ar_cap]
        ctc_g = g if ctc_cap is None else g[:ctc_cap]
        t0 = time.time()
        ctc_rows = decode_group(ctc_g, want_ar=False)
        ar_rows = decode_group(ar_g, want_ar=True)
        dt = time.time() - t0
        ap = [(r, a) for r, a, _ in ar_rows]
        cp = [(r, c) for r, _, c in ctc_rows]
        all_ar += ap
        all_ctc += cp
        print(f"{label[:40]:40}{len(ap):7d}{cer(ap):9.4f}{exact(ap):8.1%}   "
              f"{len(cp):7d}{cer(cp):9.4f}{exact(cp):8.1%}   [{dt:.0f}s]")
        rows.append([label, len(ap), cer(ap), exact(ap), len(cp), cer(cp), exact(cp)])
        for ref, ah, ct in sorted(ar_rows,
                                  key=lambda t: editdistance.eval(t[0], t[1] or ""),
                                  reverse=True)[:SHOW_WORST]:
            print(f"    gt : {ref!r}")
            print(f"    ar : {ah!r}")
            print(f"    ctc: {ct!r}")
    if len(groups) > 1:
        print("-" * len(_HDR))
        print(f"{'ALL (micro-avg)':40}{len(all_ar):7d}{cer(all_ar):9.4f}{exact(all_ar):8.1%}   "
              f"{len(all_ctc):7d}{cer(all_ctc):9.4f}{exact(all_ctc):8.1%}")
        rows.append([f"ALL::{title}".strip(":"), len(all_ar), cer(all_ar), exact(all_ar),
                     len(all_ctc), cer(all_ctc), exact(all_ctc)])
    return rows


_csv_rows = []


def flush_csv():
    with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["group", "n_ar", "ar_cer", "ar_exact", "n_ctc", "ctc_cer", "ctc_exact"])
        w.writerows(_csv_rows)
    print("\nsaved", CSV_PATH)


print("ready -- run section 3b and/or section 4")


checkpoint: /content/drive/My Drive/tuna-ocr/checkpoints/v2_ctc_ar_sequential/best.pt
ready -- run section 3b and/or section 4


## 3b. Held-out validation split

The exact 2% `run_training` held out for `v2_ctc_ar_sequential` (same shuffle
seed, same `val_frac`, same CTC-unlearnable filter). **Not** trained on, but it
*was* used to pick `best.pt`, so read it as a validation number, not a clean
test number -- for that see section 4. `AR_SAMPLES_PER_SOURCE=None` does the
whole set (a few minutes on GPU); set an int for a quick pass.

In [17]:
AR_SAMPLES_PER_SOURCE = None       # None = whole val set; e.g. 1000 for a quick pass
CTC_SAMPLES_PER_SOURCE = None      # None = whole val set (cheap regardless)
MATCH_TRAIN_VAL_FILTER = True      # drop CTC-unlearnable val samples, exactly as run_training did

samples = list(load_dedup_manifest(dedup_manifest))
random.Random(SEED).shuffle(samples)
n_val = max(1, int(len(samples) * VAL_FRAC))
val_samples = samples[:n_val]
print(f"{len(samples)} total -> {len(val_samples)} in val split")
if MATCH_TRAIN_VAL_FILTER:
    bad = set(find_unlearnable(val_samples, model_cfg, char_vocab))
    val_samples = [s for i, s in enumerate(val_samples) if i not in bad]
    print(f"dropped {len(bad)} CTC-unlearnable -> {len(val_samples)} kept")

groups = defaultdict(list)
for s in val_samples:
    groups[s.source].append(s)
print("val samples per source:", {k: len(v) for k, v in sorted(groups.items())})

_csv_rows += evaluate_groups(
    groups, ar_cap=AR_SAMPLES_PER_SOURCE, ctc_cap=CTC_SAMPLES_PER_SOURCE,
    title="held-out val (used for checkpoint selection, NOT trained on)")
flush_csv()


404912 total -> 8098 in val split
dropped 0 CTC-unlearnable -> 8098 kept
val samples per source: {'chanrith_ocr_image_line': 1998, 'darayut_scene_text': 2085, 'deepcopy_khmer_text_recognition': 2564, 'sokheng_synthetic_v1': 1451}

=== held-out val (used for checkpoint selection, NOT trained on) ===
group                                      n_ar   AR CER   AR ex     n_ctc  CTC CER  CTC ex
-------------------------------------------------------------------------------------------


TypeError: BlockwiseARDecoder.decode_greedy_sequential() got an unexpected keyword argument 'repetition_penalty'

## 4. External test set -- fully unseen

`KiteAether/c_khmer_gemma_ocr_train` is **not** in the training mix and was
**not** used for checkpoint selection, so this is the cleanest generalization
number available.

The first cell streams a sample and **prints the first row's schema + a few
previews** (text and image size). Check them before trusting the result:

- If column auto-detection picks the wrong fields, set `EXTERNAL_TEST_IMAGE_COL`
  / `EXTERNAL_TEST_TEXT_COL` explicitly and re-run.
- If the images are full **pages/paragraphs** rather than single cropped
  **lines**, this line-recognizer will score badly for reasons unrelated to its
  quality -- it expects one text line per image (`img_height=64`, then chunked).
  In that case you'd need to line-segment first (out of scope here).

In [ ]:
# ---- external test set config ----
EXTERNAL_TEST_HUB_ID = "KiteAether/c_khmer_gemma_ocr_train"
EXTERNAL_TEST_SPLIT = "train"        # this repo's only split; still unseen by our training
EXTERNAL_TEST_N = 500               # streamed rows to evaluate (raise for a tighter estimate)
EXTERNAL_TEST_IMAGE_COL = None       # None = auto-detect; else set from the schema print below
EXTERNAL_TEST_TEXT_COL = None
EXTERNAL_TEST_SHUFFLE_BUFFER = 0     # >0 = less order-correlated sampling, slower first row
# ----------------------------------

import datasets as hfds

ds = hfds.load_dataset(EXTERNAL_TEST_HUB_ID, split=EXTERNAL_TEST_SPLIT, streaming=True)
if EXTERNAL_TEST_SHUFFLE_BUFFER > 0:
    ds = ds.shuffle(seed=0, buffer_size=EXTERNAL_TEST_SHUFFLE_BUFFER)
_it = iter(ds)
_first = next(_it)

print("first row schema:")
for k, v in _first.items():
    if isinstance(v, Image.Image):
        info = f"PIL.Image size={v.size}"
    elif isinstance(v, dict):
        info = f"dict keys={list(v)}"
    elif isinstance(v, str):
        info = f"str {v[:90]!r}"
    else:
        info = f"{type(v).__name__} {v!r}"[:100]
    print(f"  {k:24} {info}")


def _pick_cols(example):
    img_col, txt_col = EXTERNAL_TEST_IMAGE_COL, EXTERNAL_TEST_TEXT_COL
    if img_col is None:
        for k, v in example.items():
            if isinstance(v, Image.Image) or (isinstance(v, dict) and v.get("bytes")):
                img_col = k
                break
    if txt_col is None:
        for k, v in example.items():
            if k != img_col and isinstance(v, str) and v.strip():
                txt_col = k
                break
    if not img_col or not txt_col:
        raise RuntimeError(
            f"could not auto-detect columns (image={img_col}, text={txt_col}). "
            f"Set EXTERNAL_TEST_IMAGE_COL / EXTERNAL_TEST_TEXT_COL from the schema above.")
    return img_col, txt_col


def _to_pil(v):
    if isinstance(v, Image.Image):
        return v
    if isinstance(v, dict) and v.get("bytes"):
        return Image.open(io.BytesIO(v["bytes"]))
    raise TypeError(f"unhandled image value: {type(v)}")


img_col, txt_col = _pick_cols(_first)
print(f"\nusing image_col={img_col!r}  text_col={txt_col!r}")

ext_samples = []
for ex in itertools.chain([_first], itertools.islice(_it, EXTERNAL_TEST_N - 1)):
    text = ex.get(txt_col)
    if not isinstance(text, str) or not text.strip():
        continue
    pil = _to_pil(ex[img_col]).convert("RGB")
    buf = io.BytesIO()
    pil.save(buf, format="PNG")
    ext_samples.append(Sample(image_bytes=buf.getvalue(), text=text.strip(),
                              source=EXTERNAL_TEST_HUB_ID))

print(f"\nbuilt {len(ext_samples)} external test samples. previews:")
for s in ext_samples[:4]:
    with open_image(s.image_source) as im:
        print(f"  {im.size}  text={s.text[:90]!r}")

_csv_rows += evaluate_groups(
    {EXTERNAL_TEST_HUB_ID: ext_samples},
    title="external test (unseen -- not trained on, not used for selection)")
flush_csv()


README.md:   0%|          | 0.00/331 [00:00<?, ?B/s]

first row schema:
  image                    PIL.Image size=(363, 153)
  text                     str 'កញ្ចប់ កង្វះ'

using image_col='image'  text_col='text'

built 500 external test samples. previews:
  (363, 153)  text='កញ្ចប់ កង្វះ'
  (160, 48)  text='ថាមពលកើតឡើង'
  (91, 47)  text='អ~អា'
  (151, 111)  text='096'

=== external test (unseen -- not trained on, not used for selection) ===
group                                      n_ar   AR CER   AR ex     n_ctc  CTC CER  CTC ex
-------------------------------------------------------------------------------------------
KiteAether/c_khmer_gemma_ocr_train          500   0.5803   16.8%       500   0.4675   13.2%   [55s]
    gt : 'She     is        was      has been     will be'
    ar : '5 he._ia ] Noa _".2025 ben_kb e'
    ctc: '5he. _is ]Noa_has been _wilLbe'
    gt : 'សម្បូរដោយ'
    ar : 'តាឡូទ “Je<J<<<<<<<<<<<<<<<<<<'
    ctc: 'ឡយទ—ានJ'
    gt : 'គ្រប់កាលៈទេសៈទាំងអស់'
    ar : 'គ្រប់គ)សៈ 6 Sសៈខា  ⁇   ⁇   ⁇   ⁇ ) ។រស'
    ctc: 'គ្របក)ល